# Assignment: Time-Series Forecasting with Recurrent Neural Networks
## Stock Price Prediction — RNN vs LSTM vs GRU | End-to-End Pipeline

---

### Assignment Objectives

By completing this assignment you will:

1. **Explain** what sequential data is, why standard feedforward networks fail on it, and how recurrent architectures solve this.
2. **Describe** stock price data — what it contains, why it is sequential, and what makes forecasting challenging.
3. **Implement** a complete end-to-end PyTorch pipeline: data loading, preprocessing, model design (RNN, LSTM, GRU), training, evaluation.
4. **Analyze** training dynamics by visualizing loss curves and comparing convergence behavior across architectures.
5. **Experiment** with hyperparameters (sequence length, hidden size, number of layers, learning rate, model type) and document the effect of each change.
6. **Interpret** model outputs through prediction plots, error distributions, and metric comparisons across the three architectures.

### Submission Requirements

| Deliverable | Description |
|---|---|
| **Notebook** | This notebook — fully executed with all outputs visible. All code cells must run without errors. |
| **Report** | A PDF report (4-6 pages) analyzing your results. Use the figures generated here. Follow the report guide in Part 15. |
| **Saved Model** | The best model saved as `.pth` file. |

### Grading Rubric

| Category | Points | Description |
|---|---|---|
| Data Exploration & Preprocessing | 15 | Proper loading, visualization, normalization, windowing, chronological split |
| Model Architecture | 20 | Correct RNN/LSTM/GRU implementations, parameter counting, architecture explanation |
| Training & Evaluation | 25 | Training loop, loss curves, proper eval, comparison across architectures |
| Visualization & Analysis | 20 | Prediction plots, error analysis, confusion of predictions, feature analysis |
| Experiments & Report | 20 | Systematic hyperparameter experiments with analysis |

**Total: 100 points**

---

*Apeiron AI | "Boundless Possibilities, Infinite Potential"*
*© 2026 | www.aperionaiml.com*

---

# Part 1: Background — What is Sequential Data?

## 1.1 Why Order Matters

Consider these two sentences:

- *"The man bit the dog"*
- *"The dog bit the man"*

Exactly the same words — completely different meanings. **Order matters.** Any model that ignores order will treat both sentences identically — and get at least one wrong.

**Sequential data** is data where the position (or timestamp) of each element carries information:

| Domain | Example | Why Order Matters |
|---|---|---|
| Natural Language | "The stock will rise" | Word order encodes grammar and meaning |
| Stock Prices | Daily closing prices | Today's price depends on yesterday's |
| Audio / Speech | Waveform samples | Temporal patterns form phonemes and words |
| DNA | ATCGGATC… | Base-pair order encodes genetic information |
| Sensor / IoT | Temperature readings | Trends and seasonality depend on time |

## 1.2 Why MLPs Fail on Sequences

A standard feedforward network (MLP) takes a **fixed-size vector** as input and has **no concept of time or order**:

```
MLP (feedforward):                  RNN (recurrent):
x₁ ──┐                            x₁ ── [h₁] ──┐
x₂ ──┼── [FC] ── ŷ               x₂ ── [h₂] ──┼── ŷ
x₃ ──┘                            x₃ ── [h₃] ──┘
  No memory!                        ↑ hidden state ↑
                                    carries memory forward
```

**Problems with MLPs for sequences:**
- **No memory** — each input is processed independently
- **Fixed input size** — cannot handle variable-length sequences
- **Order ignored** — shuffling the input does not change the output
- **No parameter sharing** across time steps

## 1.3 Recurrent Neural Networks (RNN)

An RNN processes one element at a time and maintains a **hidden state** that serves as memory:

```
      x₁         x₂         x₃         x₄
       │          │          │          │
       ▼          ▼          ▼          ▼
   ┌──────┐  ┌──────┐  ┌──────┐  ┌──────┐
   │ RNN  │─►│ RNN  │─►│ RNN  │─►│ RNN  │──► output
   │ Cell │  │ Cell │  │ Cell │  │ Cell │
   └──────┘  └──────┘  └──────┘  └──────┘
   h₀─────►h₁─────►h₂─────►h₃─────►h₄
```

**RNN equation:**

$$a^{\langle t \rangle} = \tanh(W_{aa} \cdot a^{\langle t-1 \rangle} + W_{ax} \cdot x^{\langle t \rangle} + b_a)$$

The same weights $W_{aa}$ and $W_{ax}$ are reused at every time step — this is **parameter sharing**.

## 1.4 The Vanishing Gradient Problem

During back-propagation through time (BPTT), gradients are multiplied through many time steps. With `tanh` activations (output in [-1, 1]), gradients **shrink exponentially** — the network cannot learn long-range dependencies.

Example: to predict word 100 based on word 1, the gradient must survive 99 multiplications — it vanishes to ≈ 0.

## 1.5 LSTM — Long Short-Term Memory

LSTM adds a **cell state** (a highway for gradients) and **three gates** to control information flow:

```
┌──────────────────────────────────────────┐
│                LSTM Cell                  │
│                                          │
│   ┌─────────┐  ┌─────────┐  ┌────────┐  │
│   │ Forget  │  │  Input  │  │ Output │  │
│   │  Gate   │  │  Gate   │  │  Gate  │  │
│   │ σ(Wf)  │  │ σ(Wi)  │  │ σ(Wo) │  │
│   └────┬────┘  └────┬────┘  └───┬────┘  │
│        │            │           │        │
│   C(t-1) ──×── + ──×── C(t)    │        │
│        forget   add new     ┌───┘        │
│        old info  info       │            │
│                         tanh(C(t)) × ──► h(t)
└──────────────────────────────────────────┘
```

- **Forget gate** — what to discard from cell state
- **Input gate** — what new information to store
- **Output gate** — what to output from cell state

## 1.6 GRU — Gated Recurrent Unit

GRU is a **simplified LSTM** with only **two gates** and no separate cell state:

- **Update gate (z)** — how much of the old hidden state to keep
- **Reset gate (r)** — how much of the old hidden state to use when computing the candidate

GRU has fewer parameters than LSTM (faster to train) and often performs comparably.

## 1.7 Summary: RNN vs LSTM vs GRU

| Feature | Vanilla RNN | LSTM | GRU |
|---|---|---|---|
| Gates | 0 | 3 (forget, input, output) | 2 (update, reset) |
| Cell state | No | Yes | No (uses hidden state) |
| Parameters | Fewest | Most | Middle |
| Long-range memory | Poor | Excellent | Good |
| Training speed | Fastest | Slowest | Middle |
| Vanishing gradient | Severe | Solved | Mostly solved |

---

# Part 2: Environment Setup

We install all required libraries and set up the computing environment. `yfinance` is used to download real stock-market data directly from Yahoo Finance.

> If you are running on Google Colab, all packages except `yfinance` are pre-installed.

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  INSTALL DEPENDENCIES (run once)                                             ║
# ╚══════════════════════════════════════════════════════════════════════════════╝
%pip install torch numpy pandas matplotlib scikit-learn seaborn yfinance --quiet


In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  IMPORTS                                                                     ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import warnings, os, copy, time, math
warnings.filterwarnings('ignore')

# Reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"PyTorch version: {torch.__version__}")
print(f"Device: {device}")
if device.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")


---

# Part 3: The Stock Dataset

## About Stock Price Data

Stock market data is **inherently sequential** — each observation is tied to a specific date and depends on preceding observations. We use **daily OHLCV data** (Open, High, Low, Close, Volume).

| Column | Type | Description |
|---|---|---|
| **Open** | float | Price at market open |
| **High** | float | Highest price during the day |
| **Low** | float | Lowest price during the day |
| **Close** | float | Price at market close ← **our prediction target** |
| **Volume** | int | Number of shares traded |

We will download **Apple Inc. (AAPL)** stock data from **2018-01-01 to 2024-01-01** using the `yfinance` library. A synthetic fallback is provided in case the download fails (e.g., no internet connection).

### Why Stock Prediction is Challenging

- Stock prices are influenced by **external events** (earnings, news, policy) not captured in price alone
- Markets are **partially efficient** — obvious patterns are quickly arbitraged away
- Prices exhibit **non-stationarity** — statistical properties change over time
- **Noise-to-signal ratio** is very high

> **Note:** This assignment uses stock prediction as a pedagogical exercise for learning sequential models. Real-world trading requires far more sophisticated approaches.

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  DOWNLOAD / GENERATE STOCK DATA                                              ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

def download_stock_data(ticker='AAPL', start='2018-01-01', end='2024-01-01'):
    # Download stock data with yfinance; fall back to synthetic data.
    try:
        import yfinance as yf
        df = yf.download(ticker, start=start, end=end, progress=False)
        if len(df) > 100:
            # Flatten MultiIndex columns if present
            if isinstance(df.columns, pd.MultiIndex):
                df.columns = df.columns.get_level_values(0)
            df = df[['Open', 'High', 'Low', 'Close', 'Volume']].dropna()
            print(f"Downloaded {len(df)} rows of {ticker} data ({df.index[0].date()} to {df.index[-1].date()})")
            return df
    except Exception as e:
        print(f"yfinance download failed: {e}")

    # ---- Synthetic fallback ----
    print("Generating synthetic stock data as fallback...")
    np.random.seed(42)
    dates = pd.bdate_range(start=start, end=end)
    n = len(dates)
    # Random walk with drift + trend
    returns = np.random.normal(0.0005, 0.015, n)
    close = 170.0 * np.exp(np.cumsum(returns))
    high  = close * (1 + np.abs(np.random.normal(0, 0.008, n)))
    low   = close * (1 - np.abs(np.random.normal(0, 0.008, n)))
    opn   = close * (1 + np.random.normal(0, 0.005, n))
    volume = np.random.randint(40_000_000, 150_000_000, n)
    df = pd.DataFrame({'Open': opn, 'High': high, 'Low': low,
                        'Close': close, 'Volume': volume}, index=dates)
    print(f"Generated {len(df)} rows of synthetic data")
    return df

df = download_stock_data()
print(f"\nShape: {df.shape}")
print(f"\nFirst 5 rows:\n{df.head()}")
print(f"\nBasic statistics:\n{df.describe().round(2)}")
print(f"\nData types:\n{df.dtypes}")


In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  PRICE HISTORY VISUALIZATION                                                 ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

fig, axes = plt.subplots(3, 1, figsize=(14, 12), gridspec_kw={'height_ratios': [3, 1, 1]})
fig.suptitle("AAPL Stock — Full History", fontsize=16, fontweight='bold')

# --- Price + moving averages ---
ax = axes[0]
ax.plot(df.index, df['Close'], label='Close', linewidth=1.2, color='#2196F3')
for window, color in [(20, '#FF9800'), (50, '#4CAF50'), (200, '#F44336')]:
    ma = df['Close'].rolling(window).mean()
    ax.plot(df.index, ma, label=f'{window}-day MA', linewidth=0.9, color=color, alpha=0.8)
ax.set_ylabel('Price ($)')
ax.legend(loc='upper left')
ax.set_title('Closing Price with Moving Averages')
ax.grid(True, alpha=0.3)

# --- Volume ---
ax = axes[1]
ax.bar(df.index, df['Volume'], width=1, color='#90CAF9', alpha=0.7)
ax.set_ylabel('Volume')
ax.set_title('Trading Volume')
ax.grid(True, alpha=0.3)

# --- Daily returns ---
daily_returns = df['Close'].pct_change().dropna()
ax = axes[2]
ax.hist(daily_returns, bins=100, color='#7E57C2', alpha=0.7, edgecolor='white')
ax.axvline(0, color='red', linestyle='--', linewidth=1)
ax.set_xlabel('Daily Return')
ax.set_ylabel('Frequency')
ax.set_title(f'Daily Returns Distribution (mean={daily_returns.mean():.4f}, std={daily_returns.std():.4f})')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('asgn_fig_price_history.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Daily return statistics:\n  Mean:   {daily_returns.mean():.6f}\n  Std:    {daily_returns.std():.6f}\n  Min:    {daily_returns.min():.6f}\n  Max:    {daily_returns.max():.6f}")


---

# Part 4: Data Preprocessing

## Why Normalize?

Neural networks train faster and more stably when inputs are on a similar scale. We use **MinMaxScaler** to map prices to [0, 1].

**Critical:** the scaler must be **fit only on training data** — fitting on the full dataset leaks future information into the training process.

## Sliding Window Approach

To predict the next day's price, we feed the model a **window** of the past N days:

```
Sliding Window (seq_length=60):

Day: 1  2  3  4  ...  60  → predict Day 61
     2  3  4  5  ...  61  → predict Day 62
     3  4  5  6  ...  62  → predict Day 63
     ...
```

Each window becomes one training sample: **input** = `[price_t-59, ..., price_t]`, **target** = `price_t+1`.

## Chronological Split (NOT Random!)

For time-series data, we must **never** use random train/test splits — that would leak future data into training. Instead, we use a **chronological split**:

```
|◄────── Train (80%) ──────►|◄── Test (20%) ──►|
2018                    ~2022              2024
```

This simulates the real scenario: train on historical data, evaluate on future (unseen) data.

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  HYPERPARAMETERS — collected in one place for easy experimentation           ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

SEQ_LENGTH    = 60       # days of history used to predict next day
HIDDEN_SIZE   = 64       # neurons in recurrent layer
NUM_LAYERS    = 2        # stacked recurrent layers
BATCH_SIZE    = 32       # samples per training step
LEARNING_RATE = 0.001    # step size for optimizer
NUM_EPOCHS    = 50       # full passes through training data
DROPOUT       = 0.2      # regularization
TRAIN_RATIO   = 0.8      # chronological split

print("Hyperparameters")
print("=" * 40)
for name, val in [("SEQ_LENGTH", SEQ_LENGTH), ("HIDDEN_SIZE", HIDDEN_SIZE),
                   ("NUM_LAYERS", NUM_LAYERS), ("BATCH_SIZE", BATCH_SIZE),
                   ("LEARNING_RATE", LEARNING_RATE), ("NUM_EPOCHS", NUM_EPOCHS),
                   ("DROPOUT", DROPOUT), ("TRAIN_RATIO", TRAIN_RATIO)]:
    print(f"  {name:20s} = {val}")


In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  NORMALIZE, CREATE SEQUENCES, SPLIT, DATALOADERS                             ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

# --- Use only the Close price ---
close_prices = df['Close'].values.reshape(-1, 1)

# --- Fit scaler on training portion ONLY ---
train_size = int(len(close_prices) * TRAIN_RATIO)
scaler = MinMaxScaler(feature_range=(0, 1))
scaler.fit(close_prices[:train_size])

scaled_data = scaler.transform(close_prices)
print(f"Total data points: {len(scaled_data)}")
print(f"Train size: {train_size}, Test size: {len(scaled_data) - train_size}")
print(f"Scaled range: [{scaled_data.min():.4f}, {scaled_data.max():.4f}]")

# --- Create sliding windows ---
def create_sequences(data, seq_length):
    X, y = [], []
    for i in range(len(data) - seq_length):
        X.append(data[i : i + seq_length])
        y.append(data[i + seq_length])
    return np.array(X), np.array(y)

X, y = create_sequences(scaled_data, SEQ_LENGTH)
print(f"\nTotal sequences: {len(X)}")
print(f"X shape: {X.shape}  (samples, seq_length, features)")
print(f"y shape: {y.shape}")

# --- Chronological split ---
split_idx = train_size - SEQ_LENGTH
X_train, X_test = X[:split_idx], X[split_idx:]
y_train, y_test = y[:split_idx], y[split_idx:]
print(f"\nTrain sequences: {len(X_train)}")
print(f"Test sequences:  {len(X_test)}")

# --- Convert to tensors + DataLoaders ---
X_train_t = torch.FloatTensor(X_train)
y_train_t = torch.FloatTensor(y_train)
X_test_t  = torch.FloatTensor(X_test)
y_test_t  = torch.FloatTensor(y_test)

train_dataset = TensorDataset(X_train_t, y_train_t)
test_dataset  = TensorDataset(X_test_t, y_test_t)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False)
print(f"\nTrain batches: {len(train_loader)}, Test batches: {len(test_loader)}")


### 4.1 Visualize Sliding Windows & Train/Test Split

> **For your report:** Include this figure to illustrate the sliding-window approach and the chronological train/test split.

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  SLIDING WINDOW & TRAIN/TEST SPLIT VISUALIZATION                             ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# --- Train / Test split ---
ax = axes[0]
ax.plot(df.index[:train_size], close_prices[:train_size], label='Train', color='#2196F3')
ax.plot(df.index[train_size:], close_prices[train_size:], label='Test', color='#F44336')
ax.axvline(df.index[train_size], color='black', linestyle='--', linewidth=1.5, label='Split point')
ax.set_title('Chronological Train / Test Split', fontsize=13, fontweight='bold')
ax.set_ylabel('Price ($)')
ax.legend()
ax.grid(True, alpha=0.3)

# --- Sample sliding windows ---
ax = axes[1]
window_start_indices = [0, 50, 100, 200, 400]
colors = ['#E91E63', '#9C27B0', '#3F51B5', '#009688', '#FF9800']
for idx, color in zip(window_start_indices, colors):
    if idx + SEQ_LENGTH < len(close_prices):
        window_dates = df.index[idx : idx + SEQ_LENGTH]
        window_prices = close_prices[idx : idx + SEQ_LENGTH].flatten()
        target_date = df.index[idx + SEQ_LENGTH]
        target_price = close_prices[idx + SEQ_LENGTH, 0]
        ax.plot(window_dates, window_prices, color=color, linewidth=1.5, alpha=0.7)
        ax.scatter(target_date, target_price, color=color, s=60, zorder=5,
                   edgecolors='black', linewidth=0.5)
ax.set_title(f'Sample Sliding Windows (seq_length={SEQ_LENGTH})', fontsize=13, fontweight='bold')
ax.set_ylabel('Price ($)')
ax.set_xlabel('Date')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('asgn_fig_sliding_windows.png', dpi=150, bbox_inches='tight')
plt.show()


---

# Part 5: Model Architecture

## Architecture Diagram

Our `StockPredictor` model supports three recurrent backends — `rnn`, `lstm`, or `gru` — selected by a single string argument:

```
INPUT: (batch, seq_length, 1) — 60 days of normalized prices

 ┌─── RNN / LSTM / GRU ──────────────────────────────────────┐
 │  x₁ → [h₁] → x₂ → [h₂] → ... → x₆₀ → [h₆₀]           │
 │  hidden_size=64, num_layers=2, dropout=0.2                │
 └────────────────────────────────────────────────────────────┘
                         │
              h₆₀ (last hidden state)
                         │
              ┌─── FC Layer ───┐
              │  Linear(64, 1) │
              └────────────────┘
                         │
OUTPUT: (batch, 1) — predicted next-day price (scaled)
```

All three models share the same outer structure — only the recurrent cell differs. This makes comparison fair.

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  MODEL DEFINITION — StockPredictor (RNN / LSTM / GRU)                        ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

class StockPredictor(nn.Module):
    # Recurrent model for stock price prediction.
    # Args: model_type ('rnn','lstm','gru'), input_size, hidden_size, num_layers, dropout
    def __init__(self, model_type='lstm', input_size=1, hidden_size=64,
                 num_layers=2, dropout=0.2):
        super(StockPredictor, self).__init__()
        self.model_type = model_type.lower()
        self.hidden_size = hidden_size
        self.num_layers = num_layers

        # Select recurrent layer
        rnn_cls = {'rnn': nn.RNN, 'lstm': nn.LSTM, 'gru': nn.GRU}[self.model_type]
        self.rnn = rnn_cls(input_size=input_size, hidden_size=hidden_size,
                           num_layers=num_layers, dropout=dropout if num_layers > 1 else 0,
                           batch_first=True)

        # Fully-connected output
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        # x shape: (batch, seq_length, input_size)
        out, _ = self.rnn(x)            # out: (batch, seq_length, hidden_size)
        out = out[:, -1, :]             # take last time step: (batch, hidden_size)
        out = self.fc(out)              # (batch, 1)
        return out

# Instantiate all three models
model_types = ['rnn', 'lstm', 'gru']
models = {}
for mt in model_types:
    models[mt] = StockPredictor(model_type=mt, input_size=1,
                                 hidden_size=HIDDEN_SIZE,
                                 num_layers=NUM_LAYERS,
                                 dropout=DROPOUT).to(device)
    print(f"\n{mt.upper()} model created")
    print(models[mt])


### 5.1 Model Summary — Parameter Count Comparison

> **For your report:** Include the parameter counts for all three architectures. Explain why LSTM has the most parameters and RNN the fewest.

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  MODEL SUMMARY — PARAMETER COMPARISON                                        ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

print("=" * 60)
print(f"{'Model':<10} {'Parameters':>15} {'Recurrent Params':>18} {'FC Params':>12}")
print("=" * 60)
param_counts = {}
for mt in model_types:
    total = count_parameters(models[mt])
    rnn_params = sum(p.numel() for n, p in models[mt].named_parameters() if 'rnn' in n)
    fc_params  = sum(p.numel() for n, p in models[mt].named_parameters() if 'fc' in n)
    param_counts[mt] = total
    print(f"{mt.upper():<10} {total:>15,} {rnn_params:>18,} {fc_params:>12,}")
print("=" * 60)

# Quick visual
fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(model_types, [param_counts[m] for m in model_types],
              color=['#F44336', '#2196F3', '#4CAF50'])
ax.set_title('Trainable Parameters by Architecture', fontsize=13, fontweight='bold')
ax.set_ylabel('Number of Parameters')
for bar, mt in zip(bars, model_types):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 100,
            f'{param_counts[mt]:,}', ha='center', fontsize=11, fontweight='bold')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('asgn_fig_param_comparison.png', dpi=150, bbox_inches='tight')
plt.show()


---

# Part 6: Loss Function & Optimizer

| Component | Choice | Explanation |
|---|---|---|
| **Loss** | `MSELoss` | Mean Squared Error — standard for regression tasks. Penalizes large errors more. |
| **Optimizer** | `Adam` | Adaptive learning rate — combines momentum and RMSprop. Works well out of the box. |
| **Scheduler** | `ReduceLROnPlateau` | Reduces LR by a factor when validation loss stops improving. Prevents overshooting. |

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  LOSS, OPTIMIZERS, SCHEDULERS                                                ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

criterion = nn.MSELoss()

optimizers = {}
schedulers = {}
for mt in model_types:
    optimizers[mt] = optim.Adam(models[mt].parameters(), lr=LEARNING_RATE)
    schedulers[mt] = optim.lr_scheduler.ReduceLROnPlateau(
        optimizers[mt], mode='min', factor=0.5, patience=5, verbose=False
    )
    print(f"{mt.upper()} — Optimizer: Adam (lr={LEARNING_RATE}), Scheduler: ReduceLROnPlateau")


---

# Part 7: Training Loop

## The Training Process — Step by Step

Each **epoch** = one full pass through the entire training set.

```
For each epoch:
  ┌─────────────────────────────────────────────────────┐
  │ 1. FORWARD PASS   — model(X) → predictions          │
  │ 2. COMPUTE LOSS   — MSE(predictions, actual)         │
  │ 3. BACKWARD PASS  — loss.backward() → gradients      │
  │ 4. UPDATE WEIGHTS — optimizer.step()                  │
  │ 5. EVALUATE       — run on test set (no gradients)    │
  └─────────────────────────────────────────────────────┘
  → Save model if test loss improved (early stopping checkpoint)
```

We train **all three architectures** using identical hyperparameters so the comparison is fair.

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  TRAINING & EVALUATION FUNCTIONS                                             ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

def train_one_epoch(model, loader, criterion, optimizer):
    model.train()
    total_loss = 0
    for X_batch, y_batch in loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        preds = model(X_batch)
        loss = criterion(preds, y_batch)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total_loss += loss.item() * len(X_batch)
    return total_loss / len(loader.dataset)

def evaluate(model, loader, criterion):
    model.eval()
    total_loss = 0
    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            preds = model(X_batch)
            loss = criterion(preds, y_batch)
            total_loss += loss.item() * len(X_batch)
    return total_loss / len(loader.dataset)


In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  MAIN TRAINING LOOP — ALL THREE MODELS                                       ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

histories = {mt: {'train_loss': [], 'test_loss': [], 'lr': []} for mt in model_types}
best_states = {}
best_losses = {mt: float('inf') for mt in model_types}
training_times = {}

for mt in model_types:
    print(f"\n{'='*60}")
    print(f"  Training {mt.upper()} model")
    print(f"{'='*60}")
    start_time = time.time()
    patience_counter = 0
    max_patience = 10

    for epoch in range(NUM_EPOCHS):
        train_loss = train_one_epoch(models[mt], train_loader, criterion, optimizers[mt])
        test_loss  = evaluate(models[mt], test_loader, criterion)
        current_lr = optimizers[mt].param_groups[0]['lr']
        schedulers[mt].step(test_loss)

        histories[mt]['train_loss'].append(train_loss)
        histories[mt]['test_loss'].append(test_loss)
        histories[mt]['lr'].append(current_lr)

        # Checkpoint best
        if test_loss < best_losses[mt]:
            best_losses[mt] = test_loss
            best_states[mt] = copy.deepcopy(models[mt].state_dict())
            patience_counter = 0
            marker = " ★ best"
        else:
            patience_counter += 1
            marker = ""

        if (epoch + 1) % 10 == 0 or epoch == 0:
            print(f"  Epoch {epoch+1:3d}/{NUM_EPOCHS} | "
                  f"Train Loss: {train_loss:.6f} | Test Loss: {test_loss:.6f} | "
                  f"LR: {current_lr:.6f}{marker}")

        if patience_counter >= max_patience:
            print(f"  Early stopping at epoch {epoch+1}")
            break

    elapsed = time.time() - start_time
    training_times[mt] = elapsed
    print(f"  Finished in {elapsed:.1f}s | Best test loss: {best_losses[mt]:.6f}")


In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  LOAD BEST MODEL STATES                                                      ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

for mt in model_types:
    models[mt].load_state_dict(best_states[mt])
    print(f"{mt.upper()} — loaded best model (test loss = {best_losses[mt]:.6f})")


---

# Part 8: Training Visualization

> **For your report:** Include ALL of these training plots. Discuss:
> - How quickly did each model converge?
> - Is there a gap between train and test loss? (overfitting indicator)
> - Which architecture converged fastest? Which achieved the lowest loss?
> - When did the learning rate get reduced? Did it help?

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  TRAINING CURVES — Loss per Model + Combined Overlay                         ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

colors = {'rnn': '#F44336', 'lstm': '#2196F3', 'gru': '#4CAF50'}

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# --- Individual model plots ---
for idx, mt in enumerate(model_types):
    ax = axes[idx // 2][idx % 2] if idx < 2 else axes[1][0]
    if idx == 2:
        ax = axes[1][0]
    ax.plot(histories[mt]['train_loss'], label='Train Loss', color=colors[mt], alpha=0.8)
    ax.plot(histories[mt]['test_loss'], label='Test Loss', color=colors[mt], linestyle='--', alpha=0.8)
    ax.set_title(f'{mt.upper()} — Training & Test Loss', fontsize=12, fontweight='bold')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('MSE Loss')
    ax.legend()
    ax.grid(True, alpha=0.3)

# --- Combined overlay ---
ax = axes[1][1]
for mt in model_types:
    ax.plot(histories[mt]['test_loss'], label=f'{mt.upper()} Test', color=colors[mt], linewidth=2)
ax.set_title('Test Loss — All Models', fontsize=12, fontweight='bold')
ax.set_xlabel('Epoch')
ax.set_ylabel('MSE Loss')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('asgn_fig_training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

# --- Learning rate schedule ---
fig, ax = plt.subplots(figsize=(10, 3))
for mt in model_types:
    ax.plot(histories[mt]['lr'], label=f'{mt.upper()}', color=colors[mt], linewidth=1.5)
ax.set_title('Learning Rate Schedule', fontsize=12, fontweight='bold')
ax.set_xlabel('Epoch')
ax.set_ylabel('Learning Rate')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('asgn_fig_lr_schedule.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  TRAINING TIME COMPARISON                                                    ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(model_types, [training_times[m] for m in model_types],
              color=[colors[m] for m in model_types])
ax.set_title('Training Time by Architecture', fontsize=13, fontweight='bold')
ax.set_ylabel('Time (seconds)')
for bar, mt in zip(bars, model_types):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            f'{training_times[mt]:.1f}s', ha='center', fontsize=11, fontweight='bold')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('asgn_fig_training_time.png', dpi=150, bbox_inches='tight')
plt.show()


---

# Part 9: Test Set Evaluation

Now we evaluate all three models on the **held-out test set** — data the models have NEVER seen during training. We compute:

- **MSE** — Mean Squared Error
- **RMSE** — Root Mean Squared Error (same units as price)
- **MAE** — Mean Absolute Error
- **MAPE** — Mean Absolute Percentage Error
- **R²** — Coefficient of determination

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  TEST SET EVALUATION — ALL MODELS                                            ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

def get_predictions(model, X_tensor):
    model.eval()
    with torch.no_grad():
        preds = model(X_tensor.to(device)).cpu().numpy()
    return preds

results = {}
predictions = {}

for mt in model_types:
    preds_scaled = get_predictions(models[mt], X_test_t)
    actual_scaled = y_test

    # Inverse transform to original price scale
    preds_price  = scaler.inverse_transform(preds_scaled)
    actual_price = scaler.inverse_transform(actual_scaled)

    mse  = mean_squared_error(actual_price, preds_price)
    rmse = np.sqrt(mse)
    mae  = mean_absolute_error(actual_price, preds_price)
    mape = np.mean(np.abs((actual_price - preds_price) / actual_price)) * 100
    r2   = r2_score(actual_price, preds_price)

    results[mt] = {'MSE': mse, 'RMSE': rmse, 'MAE': mae, 'MAPE': mape, 'R2': r2}
    predictions[mt] = {'actual': actual_price.flatten(), 'predicted': preds_price.flatten()}

# Print comparison table
print("=" * 75)
print(f"{'Model':<8} {'MSE':>10} {'RMSE':>10} {'MAE':>10} {'MAPE (%)':>10} {'R²':>10}")
print("=" * 75)
for mt in model_types:
    r = results[mt]
    print(f"{mt.upper():<8} {r['MSE']:>10.4f} {r['RMSE']:>10.4f} {r['MAE']:>10.4f} "
          f"{r['MAPE']:>10.4f} {r['R2']:>10.4f}")
print("=" * 75)


---

# Part 10: Prediction Visualization

> **For your report:** Include the prediction plots. Discuss:
> - How well do the predictions track the actual prices?
> - Where do the models struggle (e.g., sharp turns, trend changes)?
> - Which model tracks best visually?

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  ACTUAL vs PREDICTED — PER MODEL                                             ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

test_dates = df.index[train_size : train_size + len(predictions['rnn']['actual'])]

fig, axes = plt.subplots(3, 1, figsize=(14, 12), sharex=True)
for idx, mt in enumerate(model_types):
    ax = axes[idx]
    ax.plot(test_dates, predictions[mt]['actual'], label='Actual', color='black', linewidth=1.5)
    ax.plot(test_dates, predictions[mt]['predicted'], label=f'{mt.upper()} Predicted',
            color=colors[mt], linewidth=1.2, alpha=0.8)
    ax.set_title(f'{mt.upper()} — Actual vs Predicted (RMSE={results[mt]["RMSE"]:.2f})',
                 fontsize=12, fontweight='bold')
    ax.set_ylabel('Price ($)')
    ax.legend()
    ax.grid(True, alpha=0.3)
axes[-1].set_xlabel('Date')
plt.tight_layout()
plt.savefig('asgn_fig_predictions_individual.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  COMBINED PREDICTION OVERLAY                                                 ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(test_dates, predictions['rnn']['actual'], label='Actual', color='black', linewidth=2)
for mt in model_types:
    ax.plot(test_dates, predictions[mt]['predicted'], label=f'{mt.upper()}',
            color=colors[mt], linewidth=1.2, alpha=0.85)
ax.set_title('All Models — Actual vs Predicted', fontsize=14, fontweight='bold')
ax.set_xlabel('Date')
ax.set_ylabel('Price ($)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('asgn_fig_predictions_combined.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  ZOOM — LAST 60 DAYS                                                         ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

zoom = 60
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(test_dates[-zoom:], predictions['rnn']['actual'][-zoom:],
        label='Actual', color='black', linewidth=2, marker='o', markersize=3)
for mt in model_types:
    ax.plot(test_dates[-zoom:], predictions[mt]['predicted'][-zoom:],
            label=f'{mt.upper()}', color=colors[mt], linewidth=1.3, alpha=0.85)
ax.set_title(f'Last {zoom} Test Days — Zoomed In', fontsize=13, fontweight='bold')
ax.set_xlabel('Date')
ax.set_ylabel('Price ($)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('asgn_fig_predictions_zoom.png', dpi=150, bbox_inches='tight')
plt.show()


---

# Part 11: Error Analysis

> **For your report:** Include the error distributions and scatter plots. Discuss:
> - Are errors normally distributed? (Good sign — no systematic bias)
> - Are there outlier predictions? When do they occur?
> - Does the model under-predict or over-predict on average?

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  ERROR DISTRIBUTION HISTOGRAMS                                               ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for idx, mt in enumerate(model_types):
    ax = axes[idx]
    errors = predictions[mt]['predicted'] - predictions[mt]['actual']
    ax.hist(errors, bins=50, color=colors[mt], alpha=0.7, edgecolor='white')
    ax.axvline(0, color='black', linestyle='--', linewidth=1)
    ax.axvline(errors.mean(), color='red', linestyle='-', linewidth=1.5, label=f'Mean={errors.mean():.2f}')
    ax.set_title(f'{mt.upper()} — Prediction Errors', fontsize=12, fontweight='bold')
    ax.set_xlabel('Error ($)')
    ax.set_ylabel('Frequency')
    ax.legend()
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('asgn_fig_error_distributions.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  PREDICTION ERROR OVER TIME                                                  ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)
for idx, mt in enumerate(model_types):
    ax = axes[idx]
    errors = predictions[mt]['predicted'] - predictions[mt]['actual']
    ax.plot(test_dates, errors, color=colors[mt], linewidth=0.8, alpha=0.7)
    ax.axhline(0, color='black', linestyle='--', linewidth=1)
    ax.fill_between(test_dates, errors, 0, alpha=0.2, color=colors[mt])
    ax.set_title(f'{mt.upper()} — Error Over Time', fontsize=12, fontweight='bold')
    ax.set_ylabel('Error ($)')
    ax.grid(True, alpha=0.3)
axes[-1].set_xlabel('Date')
plt.tight_layout()
plt.savefig('asgn_fig_error_over_time.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  SCATTER PLOT — ACTUAL vs PREDICTED (with R²)                                ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for idx, mt in enumerate(model_types):
    ax = axes[idx]
    a, p = predictions[mt]['actual'], predictions[mt]['predicted']
    ax.scatter(a, p, alpha=0.4, s=10, color=colors[mt])
    # y = x line
    lims = [min(a.min(), p.min()), max(a.max(), p.max())]
    ax.plot(lims, lims, 'k--', linewidth=1, label='y = x (perfect)')
    ax.set_title(f'{mt.upper()} (R² = {results[mt]["R2"]:.4f})', fontsize=12, fontweight='bold')
    ax.set_xlabel('Actual Price ($)')
    ax.set_ylabel('Predicted Price ($)')
    ax.legend()
    ax.grid(True, alpha=0.3)
    ax.set_aspect('equal', adjustable='box')
plt.tight_layout()
plt.savefig('asgn_fig_scatter_actual_vs_predicted.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  BEST & WORST PREDICTION DAYS                                                ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

# Use the best model for this analysis
best_mt = min(results, key=lambda m: results[m]['RMSE'])
print(f"Best model: {best_mt.upper()} (RMSE = {results[best_mt]['RMSE']:.4f})\n")

errors_abs = np.abs(predictions[best_mt]['predicted'] - predictions[best_mt]['actual'])
sorted_idx = np.argsort(errors_abs)

print("TOP 10 BEST PREDICTIONS (smallest error):")
print("-" * 55)
for i in sorted_idx[:10]:
    print(f"  {test_dates[i].date()} | Actual: ${predictions[best_mt]['actual'][i]:.2f} | "
          f"Predicted: ${predictions[best_mt]['predicted'][i]:.2f} | Error: ${errors_abs[i]:.2f}")

print(f"\nTOP 10 WORST PREDICTIONS (largest error):")
print("-" * 55)
for i in sorted_idx[-10:][::-1]:
    print(f"  {test_dates[i].date()} | Actual: ${predictions[best_mt]['actual'][i]:.2f} | "
          f"Predicted: ${predictions[best_mt]['predicted'][i]:.2f} | Error: ${errors_abs[i]:.2f}")


---

# Part 12: Architecture Comparison

> **For your report:** Include the comparison bar chart and summary table. Discuss:
> - Which model achieved the lowest RMSE? MAE? MAPE?
> - Is there a clear winner, or are results close?
> - Consider the trade-off between accuracy and training time / model size.

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  METRIC COMPARISON — GROUPED BAR CHART                                       ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

metrics_to_plot = ['RMSE', 'MAE', 'MAPE']
x = np.arange(len(metrics_to_plot))
width = 0.25

fig, ax = plt.subplots(figsize=(10, 6))
for i, mt in enumerate(model_types):
    vals = [results[mt][m] for m in metrics_to_plot]
    bars = ax.bar(x + i * width, vals, width, label=mt.upper(), color=colors[mt])
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{val:.3f}', ha='center', fontsize=9, fontweight='bold')
ax.set_xticks(x + width)
ax.set_xticklabels(metrics_to_plot)
ax.set_title('Model Comparison — Error Metrics', fontsize=14, fontweight='bold')
ax.set_ylabel('Value')
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('asgn_fig_metric_comparison.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  COMPREHENSIVE COMPARISON TABLE                                              ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

print("=" * 80)
print(f"{'Metric':<20}", end="")
for mt in model_types:
    print(f"{mt.upper():>15}", end="")
print(f"{'Best':>15}")
print("=" * 80)

all_metrics = ['MSE', 'RMSE', 'MAE', 'MAPE', 'R2']
for metric in all_metrics:
    print(f"{metric:<20}", end="")
    vals = {mt: results[mt][metric] for mt in model_types}
    for mt in model_types:
        print(f"{vals[mt]:>15.4f}", end="")
    if metric == 'R2':
        best = max(vals, key=vals.get)
    else:
        best = min(vals, key=vals.get)
    print(f"{best.upper():>15} ★")
print("=" * 80)

# Training time and params
print(f"\n{'Training Time (s)':<20}", end="")
for mt in model_types:
    print(f"{training_times[mt]:>15.1f}", end="")
best_time = min(training_times, key=training_times.get)
print(f"{best_time.upper():>15} ★")

print(f"{'Parameters':<20}", end="")
for mt in model_types:
    print(f"{param_counts[mt]:>15,}", end="")
best_param = min(param_counts, key=param_counts.get)
print(f"{best_param.upper():>15} ★")


In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  BEST MODEL SELECTION                                                        ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

best_mt = min(results, key=lambda m: results[m]['RMSE'])
print(f"╔══════════════════════════════════════════════════════╗")
print(f"║  BEST MODEL: {best_mt.upper():>5}                                ║")
print(f"║  RMSE:  {results[best_mt]['RMSE']:.4f}                               ║")
print(f"║  MAE:   {results[best_mt]['MAE']:.4f}                               ║")
print(f"║  MAPE:  {results[best_mt]['MAPE']:.4f}%                              ║")
print(f"║  R²:    {results[best_mt]['R2']:.4f}                               ║")
print(f"╚══════════════════════════════════════════════════════╝")


---

# Part 13: Future Prediction

Using the best model, we perform **recursive multi-step prediction**: feed the last 60 known days to predict day 61, then append the prediction and use days 2-61 to predict day 62, and so on.

> **Warning:** Recursive prediction accumulates errors — predictions further into the future become increasingly unreliable. This is for demonstration only.

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  RECURSIVE FUTURE PREDICTION (30 DAYS)                                       ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

best_model = models[best_mt]
best_model.eval()

# Start from last SEQ_LENGTH points in the dataset
last_sequence = scaled_data[-SEQ_LENGTH:].copy()
future_days = 30
future_preds = []

with torch.no_grad():
    for _ in range(future_days):
        seq_tensor = torch.FloatTensor(last_sequence[-SEQ_LENGTH:].reshape(1, SEQ_LENGTH, 1)).to(device)
        pred = best_model(seq_tensor).cpu().numpy()[0, 0]
        future_preds.append(pred)
        last_sequence = np.append(last_sequence, [[pred]], axis=0)

# Inverse transform
future_prices = scaler.inverse_transform(np.array(future_preds).reshape(-1, 1)).flatten()
last_known_date = df.index[-1]
future_dates = pd.bdate_range(start=last_known_date + pd.Timedelta(days=1), periods=future_days)

# Plot
fig, ax = plt.subplots(figsize=(14, 6))
history_days = 120
ax.plot(df.index[-history_days:], close_prices[-history_days:], label='Historical', color='black', linewidth=1.5)
ax.plot(test_dates, predictions[best_mt]['predicted'], label=f'{best_mt.upper()} Predicted (test)',
        color=colors[best_mt], linewidth=1.2, alpha=0.7)
ax.plot(future_dates, future_prices, label=f'Future ({future_days} days)',
        color='#FF9800', linewidth=2, linestyle='--', marker='o', markersize=4)
ax.axvline(last_known_date, color='gray', linestyle=':', linewidth=1.5, label='Last known date')
ax.set_title(f'{best_mt.upper()} — {future_days}-Day Future Forecast', fontsize=14, fontweight='bold')
ax.set_xlabel('Date')
ax.set_ylabel('Price ($)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('asgn_fig_future_prediction.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\nFuture price predictions ({best_mt.upper()}):")
for d, p in zip(future_dates, future_prices):
    print(f"  {d.date()} → ${p:.2f}")


---

# Part 14: Hyperparameter Experiments

This is the most important section for your report! Here you will systematically change **one hyperparameter at a time** and observe how it affects performance.

## Experimental Design

We will test the following variations:

| Experiment | Variable Changed | Values Tested | Everything Else |
|---|---|---|---|
| **A** | Sequence Length | 20, 40, 60, 90, 120 | Default |
| **B** | Hidden Size | 16, 32, 64, 128, 256 | Default |
| **C** | Model Type | RNN, LSTM, GRU | Default (already done above) |
| **D** | Number of Layers | 1, 2, 3 | Default |

> **For your report:** For EACH experiment, include the comparison plot, state which setting performed best, and explain WHY you think that happened.

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  HELPER: Quick training function for experiments                             ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

def quick_train(model_type='lstm', seq_length=SEQ_LENGTH, hidden_size=HIDDEN_SIZE,
                num_layers=NUM_LAYERS, lr=LEARNING_RATE, epochs=20, verbose=False):
    # Train a model quickly and return test RMSE.
    # Rebuild sequences if seq_length changed
    if seq_length != SEQ_LENGTH:
        X_q, y_q = create_sequences(scaled_data, seq_length)
        split_q = train_size - seq_length
        X_tr, X_te = X_q[:split_q], X_q[split_q:]
        y_tr, y_te = y_q[:split_q], y_q[split_q:]
    else:
        X_tr, X_te, y_tr, y_te = X_train, X_test, y_train, y_test

    loader_tr = DataLoader(TensorDataset(torch.FloatTensor(X_tr), torch.FloatTensor(y_tr)),
                           batch_size=BATCH_SIZE, shuffle=True)
    loader_te = DataLoader(TensorDataset(torch.FloatTensor(X_te), torch.FloatTensor(y_te)),
                           batch_size=BATCH_SIZE, shuffle=False)

    model = StockPredictor(model_type=model_type, input_size=1,
                            hidden_size=hidden_size, num_layers=num_layers,
                            dropout=DROPOUT).to(device)
    opt = optim.Adam(model.parameters(), lr=lr)
    crit = nn.MSELoss()
    best_loss = float('inf')

    for ep in range(epochs):
        train_one_epoch(model, loader_tr, crit, opt)
        tl = evaluate(model, loader_te, crit)
        if tl < best_loss:
            best_loss = tl
            best_state = copy.deepcopy(model.state_dict())
        if verbose and (ep + 1) % 5 == 0:
            print(f"  Epoch {ep+1}/{epochs} | Test Loss: {tl:.6f}")

    model.load_state_dict(best_state)

    # Compute RMSE in original scale
    model.eval()
    with torch.no_grad():
        preds = model(torch.FloatTensor(X_te).to(device)).cpu().numpy()
    preds_price  = scaler.inverse_transform(preds)
    actual_price = scaler.inverse_transform(y_te)
    rmse = np.sqrt(mean_squared_error(actual_price, preds_price))
    return rmse

print("quick_train() helper ready.")


### Experiment A: Sequence Length

The sequence length determines how many past days the model can see. A longer window provides more context but increases computation and may add noise.

- **Too short** (20 days): Not enough history to capture medium-term trends
- **Just right** (40-60 days): Captures monthly patterns
- **Too long** (120+ days): May include irrelevant old data; slower training

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  EXPERIMENT A: Sequence Length                                               ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

seq_lengths = [20, 40, 60, 90, 120]
rmse_seq = []
print("Experiment A: Sequence Length")
print("=" * 40)
for sl in seq_lengths:
    rmse = quick_train(seq_length=sl, epochs=20)
    rmse_seq.append(rmse)
    print(f"  seq_length={sl:>4d} → RMSE = {rmse:.4f}")

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(seq_lengths, rmse_seq, 'o-', color='#2196F3', linewidth=2, markersize=8)
ax.set_title('Experiment A: Effect of Sequence Length on RMSE', fontsize=13, fontweight='bold')
ax.set_xlabel('Sequence Length (days)')
ax.set_ylabel('Test RMSE ($)')
for x, y in zip(seq_lengths, rmse_seq):
    ax.annotate(f'{y:.2f}', (x, y), textcoords="offset points", xytext=(0, 10), ha='center')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('asgn_fig_exp_seq_length.png', dpi=150, bbox_inches='tight')
plt.show()


### Experiment B: Hidden Size

The hidden size controls the model's **capacity** — how much information it can store in its internal representation.

- **Too small** (16): Underfitting — cannot capture complex patterns
- **Just right** (64): Good balance of capacity and generalization
- **Too large** (256): Overfitting risk — memorizes noise; much slower

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  EXPERIMENT B: Hidden Size                                                   ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

hidden_sizes = [16, 32, 64, 128, 256]
rmse_hidden = []
print("Experiment B: Hidden Size")
print("=" * 40)
for hs in hidden_sizes:
    rmse = quick_train(hidden_size=hs, epochs=20)
    rmse_hidden.append(rmse)
    print(f"  hidden_size={hs:>4d} → RMSE = {rmse:.4f}")

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(hidden_sizes, rmse_hidden, 's-', color='#4CAF50', linewidth=2, markersize=8)
ax.set_title('Experiment B: Effect of Hidden Size on RMSE', fontsize=13, fontweight='bold')
ax.set_xlabel('Hidden Size')
ax.set_ylabel('Test RMSE ($)')
for x, y in zip(hidden_sizes, rmse_hidden):
    ax.annotate(f'{y:.2f}', (x, y), textcoords="offset points", xytext=(0, 10), ha='center')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('asgn_fig_exp_hidden_size.png', dpi=150, bbox_inches='tight')
plt.show()


### Experiment C: Model Type Comparison

This experiment was already completed in Parts 7-12 above. Refer to the metric comparison table and bar chart.

**Key reference values:**

| Model | RMSE | MAE | MAPE | R² |
|---|---|---|---|---|
| RNN | (see Part 9) | — | — | — |
| LSTM | (see Part 9) | — | — | — |
| GRU | (see Part 9) | — | — | — |

> Fill in the actual values from your run for the report.

### Experiment D: Number of Layers

Stacking recurrent layers lets the model learn **hierarchical temporal features**:

- **1 layer**: Simple patterns only
- **2 layers**: Can capture multi-scale temporal features
- **3+ layers**: Diminishing returns; harder to train; dropout becomes critical

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  EXPERIMENT D: Number of Layers                                              ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

num_layers_list = [1, 2, 3]
rmse_layers = []
print("Experiment D: Number of Layers")
print("=" * 40)
for nl in num_layers_list:
    rmse = quick_train(num_layers=nl, epochs=20)
    rmse_layers.append(rmse)
    print(f"  num_layers={nl} → RMSE = {rmse:.4f}")

fig, ax = plt.subplots(figsize=(8, 5))
ax.bar([str(n) for n in num_layers_list], rmse_layers, color=['#FF9800', '#2196F3', '#9C27B0'])
ax.set_title('Experiment D: Effect of Number of Layers on RMSE', fontsize=13, fontweight='bold')
ax.set_xlabel('Number of Recurrent Layers')
ax.set_ylabel('Test RMSE ($)')
for i, (n, r) in enumerate(zip(num_layers_list, rmse_layers)):
    ax.text(i, r + 0.1, f'{r:.2f}', ha='center', fontsize=11, fontweight='bold')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('asgn_fig_exp_num_layers.png', dpi=150, bbox_inches='tight')
plt.show()


---

# Part 15: Save Model & Conclusion

## 15.1 Save the Trained Model

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  SAVE THE BEST MODEL                                                         ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

save_path = f'best_stock_model_{best_mt}.pth'
torch.save({
    'model_type': best_mt,
    'model_state_dict': models[best_mt].state_dict(),
    'hidden_size': HIDDEN_SIZE,
    'num_layers': NUM_LAYERS,
    'seq_length': SEQ_LENGTH,
    'scaler_min': scaler.data_min_[0],
    'scaler_max': scaler.data_max_[0],
    'results': results[best_mt]
}, save_path)

print(f"Model saved to: {save_path}")
print(f"Model type: {best_mt.upper()}")
print(f"File size: {os.path.getsize(save_path) / 1024:.1f} KB")


## 15.2 Summary of All Saved Figures

Every figure generated by this notebook is saved as a PNG for your report:

| Figure File | Content | Report Section |
|---|---|---|
| `asgn_fig_price_history.png` | Price chart, volume, moving averages, returns | Dataset |
| `asgn_fig_sliding_windows.png` | Sliding window visualization and train/test split | Methodology |
| `asgn_fig_param_comparison.png` | Parameter count bar chart for RNN/LSTM/GRU | Architecture |
| `asgn_fig_training_curves.png` | Training & test loss curves for all models | Training |
| `asgn_fig_lr_schedule.png` | Learning rate schedule over epochs | Training |
| `asgn_fig_training_time.png` | Training time comparison | Training |
| `asgn_fig_predictions_individual.png` | Actual vs predicted for each model | Results |
| `asgn_fig_predictions_combined.png` | All models overlaid on actual prices | Results |
| `asgn_fig_predictions_zoom.png` | Zoomed-in view of last 60 test days | Results |
| `asgn_fig_error_distributions.png` | Error histograms for all models | Analysis |
| `asgn_fig_error_over_time.png` | Prediction error over time | Analysis |
| `asgn_fig_scatter_actual_vs_predicted.png` | Scatter plots with R² | Analysis |
| `asgn_fig_metric_comparison.png` | Grouped bar chart (RMSE, MAE, MAPE) | Comparison |
| `asgn_fig_future_prediction.png` | 30-day future forecast | Future Prediction |
| `asgn_fig_exp_seq_length.png` | Experiment A results | Experiments |
| `asgn_fig_exp_hidden_size.png` | Experiment B results | Experiments |
| `asgn_fig_exp_num_layers.png` | Experiment D results | Experiments |

## 15.3 Report Writing Guide

### What to Write in Each Section

**1. Introduction (1 page)**
- What is sequential data? What are recurrent neural networks?
- Why are RNNs better than MLPs for time-series data?
- Explain the difference between RNN, LSTM, and GRU — gates, cell state, vanishing gradients
- Include the architecture diagram from Part 1

**2. Dataset (0.5 page)**
- Describe the stock data: source (AAPL), date range, OHLCV features
- Include `asgn_fig_price_history.png`
- Discuss trends, volatility, and any notable events visible in the chart

**3. Methodology (1 page)**
- Preprocessing: normalization (why?), sliding windows (how?), chronological split (why not random?)
- Model architecture: include the architecture diagram, parameter count comparison table
- Training setup: loss function (MSE), optimizer (Adam), scheduler, early stopping

**4. Results (1.5 pages)**
- Include `asgn_fig_predictions_combined.png` and `asgn_fig_predictions_zoom.png`
- Include the metric comparison table (RMSE, MAE, MAPE, R² for all three models)
- Include `asgn_fig_metric_comparison.png`
- Discuss: which model performed best? Is the difference significant?

**5. Analysis (1 page)**
- Include `asgn_fig_error_distributions.png` — are errors normally distributed?
- Include `asgn_fig_scatter_actual_vs_predicted.png` — how close to the y=x line?
- Discuss best/worst prediction days — what happened on those dates?
- Include `asgn_fig_training_curves.png` — convergence speed comparison

**6. Experiments (1 page)**
- For EACH experiment: include the comparison plot and summary
- State your observation and explain WHY you think that happened
- Recommend the best setting for each hyperparameter

**7. Conclusion (0.5 page)**
- Summarize key findings: which architecture is best and why
- Limitations of this approach (univariate, no external features, recursive prediction drift)
- Potential improvements: multivariate input, attention mechanisms, transformer models

---

### End of Notebook

Congratulations! You have completed a full end-to-end sequential modeling pipeline. You now have all the figures, data, and analysis needed to write an excellent report.

---

*Apeiron AI | "Boundless Possibilities, Infinite Potential"*
*© 2026 | www.aperionaiml.com*